# 02 — Data dictionary

Describes **every column in every file** of the Drive folder *Darb DataSets* (MC, Gastat, Monshaat, Sama: 27 files).

**Phase 1 (this notebook now): facts, measured by code.** For each column: its original name, type, missing share,
number of different values, min/max, format pattern and most common values.
**Phase 2 (later): written details.** Meaning (English + Arabic), source, unit, quality issues, role, join key and leakage risk.

**Rules for this notebook**
- The raw files are never changed. Results are saved as new files in the data folder, never in Git.
- No private values are shown: business names and IDs appear only as patterns (for example `9×10` = 10 digits).
- Clear all outputs before committing.

### Cell 1 — Setup
Finds the data folder. On Colab the files are downloaded into `/content/darb-data` (temporary copies, gone when
the session ends). On a laptop it reads a local copy of the Drive folder.
`%pip` is used instead of `!pip` because `!pip` can hang in the VS Code Colab extension.

In [ ]:
import os
import gc
import re
from pathlib import Path

import pandas as pd

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False

if IN_COLAB:
    %pip install -q gdown python-calamine
    DATA_DIR = Path('/content/darb-data')
else:
    # local copy of the Drive folder "Darb DataSets" (outside Git and OneDrive)
    DATA_DIR = Path(os.environ.get('DARB_DATA', str(Path.home() / 'darb-data')))

print('Running on Colab' if IN_COLAB else 'Running locally', '| data folder:', DATA_DIR)

### Cell 2 — Every file in Darb DataSets
The full list of the 27 files, with the folder they are in, the period they cover (when the name says it)
and their Google Drive ID. Each file is downloaded once into `DATA_DIR/<folder>/`, the same layout as
`01_preprocessing`, so both notebooks find the same files.

**Why a fixed list:** downloading the whole folder at once is limited to 50 files and depends on how the folder
is shared. Downloading by ID works the same way `01_preprocessing` already does.

In [ ]:
QUARTERS = ['2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1', '2026Q2', '2026Q3']

# (folder, period, file name, Google Drive ID)
FILES = [
    # Ministry of Commerce: active commercial registrations, one row per registration (2 parts per quarter)
    ('MC', '2024Q3', 'CR_2024-03q-active-crs_PART1-2.xlsx', '1YAjKdTEzTf66Z_wK0nNVAboD_NoskbKV'),
    ('MC', '2024Q3', 'CR_2024-03q-active-crs_PART2-2.xlsx', '1bCd3HFyhCk9VbZgT79wxsIxgBBflKps0'),
    ('MC', '2024Q4', 'CR_2024-04q-active-crs_PART1.xlsx',   '1GPJUCZ5HiTa-_ZJBVLjrp58Ha53nOBGN'),
    ('MC', '2024Q4', 'CR_2024-04q-active-crs_PART2.xlsx',   '10y-NeQI6P78Sb_b3mqjTjcdpZq9zNvNK'),
    ('MC', '2025Q1', 'CR_2025-01q-active-crs_PART1.xlsx',   '1QTeifZpbmotbWmbZoliWoVcDxSK8y8uf'),
    ('MC', '2025Q1', 'CR_2025-01q-active-crs_PART2.xlsx',   '12LqZsJ8WHI4qUza_mXP4iUQPuwG9x_2d'),
    ('MC', '2025Q2', '2025-02q-active-crs_PART1-2.xlsx',    '1pAlWmlVTGwCY2XnwhuujvFz9w8ys3nfp'),
    ('MC', '2025Q2', '2025-02q-active-crs_PART2.xlsx',      '14LNdY7UCE-itRV4ubyfRzapGtwgoDda_'),
    ('MC', '2025Q3', '2025-03q-active-crs_PART1.xlsx',      '11TR0gdOc1u486yR8y9jYkA5jXqwLwxVo'),
    ('MC', '2025Q3', '2025-03q-active-crs_PART2-2.xlsx',    '1K4jyfIVbWfALc8bdB0-agPAwBev6SqFD'),
    ('MC', '2025Q4', '2025-04q-active-crs_PART1.xlsx',      '1tzttjeDaz8WQrFSGE7WYI5AMQiNSsRrM'),
    ('MC', '2025Q4', '2025-04q-active-crs_PART2.xlsx',      '1rqw-5VqvreNlA7hJN4sY3wOzgmOI485a'),
    ('MC', '2026Q1', '2026 01q active_PART1.xlsx',          '1uXVoAo8IoDVhZEBJcZEdq1T3Rhfkw9kZ'),
    ('MC', '2026Q1', '2026 01q active_PART2.xlsx',          '1YukIJGFb8GIBxV5rCxXmXN023kU6bjr2'),
    ('MC', '2026Q2', '2026 02q active crs_PART1.xlsx',      '1KBjgL6FXdI-aeWumy15rnhlE20_W0ZNJ'),
    ('MC', '2026Q2', '2026 02q active crs_PART2.xlsx',      '1BrmeQHLyxET9xPulJDxBQ_68qZ8zu6kf'),
    ('MC', '2026Q3', 'CR A 2026-09-30_PART1.xlsx',          '1TGnGo3x2rkvDvbwREznik4ko0WGzXhHy'),
    ('MC', '2026Q3', 'CR A 2026-09-30_PART2-2.xlsx',        '1siJD5yNZTXRINRazJP2LfpFm8xlKcVZw'),
    # Gastat (General Authority for Statistics)
    ('Gastat', '', 'gastat_gdp_2026-10-21T09_14_09.657Z.csv',                              '1fRoZuqgt1ltaQcXfSzbI2rk9mjK1pRH8'),
    ('Gastat', '', 'gastat_sme_operating_expenses_revenues_2026-10-21T09_39_24.273Z.csv',  '11WopaP1bfkU4bmeugGygvgxjorm56iE-'),
    ('Gastat', '', 'National Classification for the Economic Activities(SSIC)_15 Sep_2026_AR.xlsx', '1wlS-84Ojsiv0UP8sjmZmNcENU_M8J0wZ'),
    # Monshaat (Small and Medium Enterprises General Authority)
    ('Monshaat', '2026Q1', 'Number of Enterprises 2026 Q1 Excel.xlsx',     '1Lf9dn8wJMHHLi5ihrpUX4Sm2VbWGD5tc'),
    ('Monshaat', '2026Q2', 'Number of Enterprises 2026 Q2 Excel.xlsx',     '1oAZnhNgkL2uOdiH2QOzDAqqxObZVPMEt'),
    ('Monshaat', '2026Q1', 'Number of New Enterprises 2026 Q1 Excel.xlsx', '10GX9fN7wuq0Scp26eeg7gliBcHKO0eKv'),
    ('Monshaat', '2026Q2', 'Number of New Enterprises 2026 Q2 Excel.xlsx', '18zU6D-INN1-a3F1ND6hRRjdI4ezzckH1'),
    # Sama (Saudi Central Bank)
    ('Sama', '', 'point-of-sale-transactions-by-sector-and-city.xlsx',  '1iMoJekCkgIWCva3-H1RPxpGBJQ64oNz3'),
    ('Sama', '', 'points-of-sale-transactions-and-sales-by-sector.xlsx', '1ZZY1XCnTcFUjegFPrGvcWzIq1yW2OcQj'),
]

def path_of(folder, name):
    return DATA_DIR / folder / name

if IN_COLAB:
    import gdown
    for folder, _, name, drive_id in FILES:
        path = path_of(folder, name)
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            print('downloading', folder, name)
            gdown.download(id=drive_id, output=str(path), quiet=True)

files = pd.DataFrame(FILES, columns=['folder', 'period', 'file', 'drive_id']).drop(columns='drive_id')
files['MB'] = [round(path_of(f, n).stat().st_size / 1e6, 1) if path_of(f, n).exists() else None
               for f, n in zip(files['folder'], files['file'])]
print(files.to_string())
print()
print(f'{files["MB"].notna().sum()} of {len(files)} files found | missing:',
      files.loc[files['MB'].isna(), 'file'].tolist() or 'none')

### Cell 3 — How each table starts (Gastat, Monshaat, Sama)
The MC files start with the column names in the first row. The statistics files often start with a title, notes
or empty rows, so the code guesses the **header row**: the first row that is (almost) as full as the fullest row
near the top. This cell prints the first rows of every sheet with the guessed header row, so we can check it.

**If a guess is wrong**, add the right row to `HEADER_ROW` (row numbers as printed on the left) and run this cell
again. These files hold only totals, not businesses, so showing their rows is safe. MC rows are never printed.

In [ ]:
# Fixes for wrong header guesses: {(file name, sheet name): header row}
HEADER_ROW = {}

def read_raw(path):
    """Every sheet of a file as text, without a header: {sheet name: table}."""
    if path.suffix.lower() == '.csv':
        for encoding in ['utf-8-sig', 'cp1256']:
            try:
                return {'(csv)': pd.read_csv(path, header=None, dtype=str, encoding=encoding)}
            except UnicodeDecodeError:
                continue
        raise ValueError(f'Could not read {path.name}')
    return pd.read_excel(path, engine='calamine', header=None, dtype=str, sheet_name=None)

def find_header(raw):
    """First row near the top that is at least 80% as full as the fullest row there (titles have 1 cell)."""
    top = raw.head(25).notna().sum(axis=1)
    return int(top.index[top >= 0.8 * top.max()][0]) if len(top) and top.max() > 0 else 0

def with_header(raw, header_row):
    """Use one row as column names; the rows below it are the data."""
    names, seen = [], {}
    for position, value in enumerate(raw.iloc[header_row], 1):
        value = '' if pd.isna(value) else re.sub(r'\s+', ' ', str(value)).strip()
        value = value or f'(no name, column {position})'
        seen[value] = seen.get(value, -1) + 1
        names.append(value if seen[value] == 0 else f'{value}.{seen[value]}')
    table = raw.iloc[header_row + 1:].copy()
    table.columns = names
    table = table.dropna(how='all')
    unnamed_and_empty = [c for c in table.columns if c.startswith('(no name') and table[c].isna().all()]
    return table.drop(columns=unnamed_and_empty)

with pd.option_context('display.max_columns', 40, 'display.width', 250, 'display.max_colwidth', 30):
    for folder, _, name, _ in FILES:
        if folder == 'MC':
            continue
        for sheet, raw in read_raw(path_of(folder, name)).items():
            header = HEADER_ROW.get((name, sheet), find_header(raw))
            print('=' * 100)
            print(f'{folder} | {name} | sheet: {sheet} | {raw.shape[0]:,} rows x {raw.shape[1]} columns | header row guess: {header}')
            print(raw.head(header + 6).fillna('').to_string())
            print()

### Cell 4 — Profile every column of every file
For each column of each sheet of each file, the code measures:

| Field | Meaning |
|---|---|
| `kind` | number, date, text, or empty (what at least 95% of the filled values look like) |
| `missing %` | share of empty cells |
| `unique` | number of different values |
| `min` / `max` | for numbers and dates |
| `top values` | the 10 most common values with their share (never for names and IDs) |
| `patterns` | the 3 most common shapes: `9×10` = 10 digits, `ع` = an Arabic word, `a` = a Latin word |
| `common name` | MC only: the shared name used in `01_preprocessing` (Cell 8), e.g. `id` for both ID columns |

**Why:** these facts come from the data, not from guessing, and the cell can be run again whenever the data changes.
The MC files are large (about 1 million rows each), so this cell takes a few minutes. Patterns are measured on a
random 200,000 values per column; every other number uses all rows.

In [ ]:
# Same mapping as 01_preprocessing Cell 8 (keep the two in sync)
COMMON_NAMES = {
    'الرقم الوطني': 'id', 'الرقم الموحد': 'id',
    'الرقم الرئيسي للسجل التجاري': 'main_id',
    'رقم السجل': 'cr_number',
    'نوع السجل التجاري': 'record_type', 'نوع السجل': 'record_type',
    'اسم السجل التجاري': 'name', 'اسم السجل': 'name',
    'نوع الكيان': 'entity_type', 'الكيان القانوني': 'entity_type',
    'نوع النشاط التجاري (بالعربية)': 'legal_form', 'الكيان القانوني.1': 'legal_form',
    'رأس المال': 'capital',
    'اسم المدينة (بالعربية)': 'city', 'المدينة': 'city',
    'اسم المنطقة (بالعربية)': 'region', 'المنطقة': 'region',
    'مفصح بالتجارة الإلكترونية؟': 'ecommerce', 'مفصح بتجارة إلكترونية؟': 'ecommerce', 'مفصح تجارة إلكترونية؟': 'ecommerce',
    'تاريخ الإصدار (ميلادي)': 'start_date', 'تاريخ انشاء السجل': 'start_date',
}
PRIVATE = {'id', 'main_id', 'cr_number', 'name'}   # their values are never shown, only their patterns
PATTERN_SAMPLE = 200_000

def shape(values):
    """Replace digits with 9×n, Arabic words with ع and Latin words with a, so no real value is shown."""
    out = values.str.replace(r'\d+', lambda m: '9' if len(m.group()) == 1 else f'9×{len(m.group())}', regex=True)
    out = out.str.replace(r'[ء-ْٱ-ۓ]+', 'ع', regex=True)
    out = out.str.replace(r'[A-Za-z]+', 'a', regex=True)
    return out.str.replace(r'\s+', ' ', regex=True)

def share_list(counts, total, limit):
    return ' | '.join(f'{value} ({100 * count / total:.1f}%)' for value, count in counts.head(limit).items())

def profile_column(column, common):
    values = column.astype('string').str.strip()
    values = values.mask(values == '')
    filled = values.dropna()
    row = {'rows': len(values),
           'missing %': round(100 * (1 - len(filled) / len(values)), 1) if len(values) else None,
           'unique': int(filled.nunique())}
    if filled.empty:
        return {**row, 'kind': 'empty'}
    sample = filled.sample(min(len(filled), PATTERN_SAMPLE), random_state=0)
    numbers = pd.to_numeric(filled.str.replace(',', '', regex=False), errors='coerce')
    if numbers.notna().mean() >= 0.95:
        row['kind'] = 'number'
        if common not in PRIVATE:
            row['min'], row['max'] = numbers.min(), numbers.max()
    elif sample.str.match(r'^\d{4}-\d{2}-\d{2}').mean() >= 0.95:
        row['kind'] = 'date'
        row['min'], row['max'] = filled.str[:10].min(), filled.str[:10].max()
    else:
        row['kind'] = 'text'
    if common not in PRIVATE and (row['kind'] == 'text' or row['unique'] <= 20):
        row['top values'] = share_list(filled.value_counts(), len(filled), 10)
    row['patterns'] = share_list(shape(sample).value_counts(), len(sample), 3)
    return row

inventory, profile = [], []
for folder, period, name, _ in FILES:
    path = path_of(folder, name)
    if folder == 'MC':
        sheets = pd.read_excel(path, engine='calamine', dtype=str, sheet_name=None)
        headers = {sheet: 0 for sheet in sheets}
    else:
        sheets, headers = {}, {}
        for sheet, raw in read_raw(path).items():
            headers[sheet] = HEADER_ROW.get((name, sheet), find_header(raw))
            sheets[sheet] = with_header(raw, headers[sheet])
    for sheet, table in sheets.items():
        table.columns = [str(c) for c in table.columns]
        inventory.append({'folder': folder, 'period': period, 'file': name, 'sheet': sheet,
                          'header row': headers[sheet], 'rows': len(table), 'columns': table.shape[1]})
        for position, column in enumerate(table.columns, 1):
            common = COMMON_NAMES.get(column, '') if folder == 'MC' else ''
            profile.append({'folder': folder, 'period': period, 'file': name, 'sheet': sheet,
                            'position': position, 'column': column, 'common name': common,
                            **profile_column(table[column], common)})
        print(f'{folder:<9} {period:<7} {name[:60]:<60} | {sheet[:25]:<25} | {len(table):>10,} rows | {table.shape[1]:>3} columns')
    del sheets
    gc.collect()

inventory = pd.DataFrame(inventory)
profile = pd.DataFrame(profile)
ORDER = ['folder', 'period', 'file', 'sheet', 'position', 'column', 'common name', 'kind', 'rows',
         'missing %', 'unique', 'min', 'max', 'top values', 'patterns']
profile = profile.reindex(columns=ORDER)
print()
print(f'{len(inventory)} sheets in {inventory["file"].nunique()} files | {len(profile):,} columns profiled')

### Cell 5 — The profile, file by file
MC columns are shown once per quarter (both parts together, PART1 first) so the list stays readable; the full
list with every file is in the saved Excel file (Cell 7). Long text is cut to 120 characters here only.

In [ ]:
def show(table):
    cut = table.copy()
    for column in ['top values', 'patterns']:
        cut[column] = cut[column].fillna('').str.slice(0, 120)
    with pd.option_context('display.max_columns', 30, 'display.width', 300, 'display.max_colwidth', 120):
        print(cut.drop(columns=['folder', 'rows']).fillna('').to_string(index=False))

print('MC: one file per quarter (PART1)')
first_parts = profile[(profile['folder'] == 'MC')].drop_duplicates(['period', 'column'], keep='first')
show(first_parts)
for folder in ['Gastat', 'Monshaat', 'Sama']:
    print()
    print('=' * 100)
    print(folder)
    show(profile[profile['folder'] == folder])

### Cell 6 — Which columns appear in which file
**MC:** one row per column, ✅ for each quarter it appears in. This shows renamed columns (for example the ID column:
`الرقم الوطني` up to 2025 Q3, `الرقم الموحد` from 2025 Q4) and columns that exist only in some years.
It also lists columns found in only one of the two parts of a quarter.
**Gastat, Monshaat, Sama:** the same check across the files of each folder (for example Q1 vs Q2 of Monshaat).

In [ ]:
mc_profile = profile[profile['folder'] == 'MC']
mc_by_quarter = (mc_profile.assign(seen='✅')
                 .pivot_table(index=['common name', 'column'], columns='period', values='seen', aggfunc='first')
                 .reindex(columns=QUARTERS).fillna('—'))
print('MC columns by quarter:')
print(mc_by_quarter.to_string())

parts_per_quarter = mc_profile.groupby('period')['file'].nunique()
parts_with_column = mc_profile.groupby(['period', 'column'])['file'].nunique()
one_part_only = parts_with_column[parts_with_column < parts_per_quarter.reindex(parts_with_column.index.get_level_values('period')).to_numpy()]
print()
print('MC columns found in only one part of a quarter:', one_part_only.index.tolist() or 'none')

other_by_file = {}
for folder in ['Gastat', 'Monshaat', 'Sama']:
    part = profile[profile['folder'] == folder]
    other_by_file[folder] = (part.assign(seen='✅', where=part['file'] + ' | ' + part['sheet'])
                             .pivot_table(index='column', columns='where', values='seen', aggfunc='first').fillna('—'))
    print()
    print(f'{folder} columns by file and sheet:')
    with pd.option_context('display.width', 300, 'display.max_colwidth', 60):
        print(other_by_file[folder].to_string())

### Cell 7 — Save the Phase 1 profile
Saves `data_dictionary_profile.xlsx` in the data folder (never in Git), with the sheets `inventory`, `columns`,
`mc_by_quarter` and one sheet per other folder.

In [ ]:
OUT = DATA_DIR / 'data_dictionary_profile.xlsx'
with pd.ExcelWriter(OUT) as writer:
    inventory.to_excel(writer, sheet_name='inventory', index=False)
    profile.to_excel(writer, sheet_name='columns', index=False)
    mc_by_quarter.to_excel(writer, sheet_name='mc_by_quarter')
    for folder, table in other_by_file.items():
        table.to_excel(writer, sheet_name=f'{folder.lower()}_by_file')
print('Saved', OUT)

## Phase 2 — Written details

Phase 1 measured the facts. Phase 2 adds what code cannot know: what each column **means**, how it can be used,
and what can go wrong. Written from the Phase 1 results (Cells 3, 5 and 6).

**Fields**

| Field | Meaning |
|---|---|
| `meaning (English)` / `المعنى (عربي)` | what the column represents |
| `unit / format` | e.g. SAR, count, date, 10-digit ID |
| `quality issues` | problems found in Phase 1 |
| `role` | **ID** (links records), **filter** (chooses rows), **helper** (used to build something else), **feature** (candidate model input), **label**, **reference** (lookup table), **note** (not data), **not for model** |
| `join key` | how the column attaches to one business |
| `leakage risk` | can the value carry information from *after* the business started? |

**The leakage rule used everywhere:** a feature may only use information available **before or at the business's
start quarter**. For MC columns this means the value from the business's **first quarter**, not a later one
(a later value can reflect what happened to the business). For outside data (Gastat, Sama) it means the latest
period that ends **before** the start quarter.

### Cell 8 — Description of every column
MC columns are described once per **common name** (they appear in 9 quarters under different Arabic names).
The other files are described column by column. `file = *` means the same description holds for every file of
that folder (the 4 Monshaat files have identical columns).

In [ ]:
DESCRIPTIONS = []

def describe(folder, file, column, en, ar, unit, issues, role, join, leakage):
    DESCRIPTIONS.append({'folder': folder, 'file': file, 'column': column,
                         'meaning (English)': en, 'المعنى (عربي)': ar, 'unit / format': unit,
                         'quality issues': issues, 'role': role, 'join key': join, 'leakage risk': leakage})

ALL = '*'
FROM_START = 'Low if taken from the business\'s first quarter; a value from a later quarter may reflect what happened after the start.'

# ---------------------------------------------------------------- MC (one row per common name)
MC = 'MC'
describe(MC, ALL, 'id',
    'Record ID: the national number up to 2025 Q3 (الرقم الوطني), the unified number from 2025 Q4 (الرقم الموحد). Same numbers, only the column name changed (01_preprocessing Part 1).',
    'رقم السجل: الرقم الوطني حتى الربع الثالث 2025، ثم الرقم الموحد من الربع الرابع 2025. الأرقام نفسها، تغيّر اسم العمود فقط.',
    '10 digits', 'Missing for all of 2025 Q3 PART1 (recovered for 83% in 01_preprocessing Cell 13b). One ID repeats in 2025 Q2.',
    'ID', 'Links the same business across quarters (becomes entity_id)', 'None (not a feature)')
describe(MC, ALL, 'main_id',
    'For a branch: the ID of its main record. Empty for main records.',
    'للفرع: رقم السجل الرئيسي التابع له. فارغ للسجلات الرئيسية.',
    '10 digits', 'Only up to 2025 Q3; the column does not exist from 2025 Q4, so branches cannot be linked to their main record after that.',
    'helper', 'main_id → id of the main record', 'High if used to count branches opened after the start')
describe(MC, ALL, 'cr_number',
    'Commercial registration number (a different number from the unified ID).',
    'رقم السجل التجاري (رقم مختلف عن الرقم الموحد).',
    '10 digits', 'Only from 2025 Q4. 0.8% missing in 2026 Q3.',
    'helper', 'None used', 'None (not a feature)')
describe(MC, ALL, 'record_type',
    'Main record (رئيسي) or branch (فرعي).',
    'نوع السجل: رئيسي أو فرعي.',
    'category: رئيسي / فرعي', 'The branch share differs a lot between files and parts (0.7% to 44%). 2026 Q2 PART1 has the header row repeated as a data row ("نوع السجل"). Missing in 2025 Q3 PART1 (copied from 2025 Q2 in Cell 13c).',
    'filter', 'Only main records are used as businesses', 'None (filter)')
describe(MC, ALL, 'name',
    'Registered business name.',
    'الاسم التجاري المسجل.',
    'Arabic text, usually 4–6 words', 'Private: never shown. Names change over time for the same ID.',
    'helper', 'Used to build ssic_* (sector) and to link old and new IDs', FROM_START)
describe(MC, ALL, 'entity_type',
    'Type of entity: establishment (مؤسسة) or company (شركة); from 2025 Q2 also branches of foreign or Gulf companies.',
    'نوع الكيان: مؤسسة أو شركة، ومن الربع الثاني 2025 أيضاً فروع الشركات الأجنبية والخليجية.',
    'category, 2–6 values', 'In 2025 Q4 the column "الكيان القانوني" holds the detailed legal form (32 values) instead, so 2025 Q4 has no real entity type. 2026 Q2 has the header row as a value.',
    'feature', 'Own column of the business', FROM_START)
describe(MC, ALL, 'legal_form',
    'Detailed legal form (e.g. sole establishment, LLC, one-person LLC, joint-stock). Despite the Arabic name "نوع النشاط التجاري" it is not the business activity.',
    'الشكل القانوني التفصيلي (مؤسسة فردية، ذات مسؤولية محدودة، شخص واحد، مساهمة...). رغم اسمه "نوع النشاط التجاري" فهو ليس النشاط.',
    'category, 13–48 values', 'Three column names over time and missing in 2025 Q4 (its values are in entity_type there). Spelling changed in 2025 Q2 ("مؤســـسة فـــردية" → "مؤسسة"), so values need one common list.',
    'feature', 'Own column of the business', FROM_START)
describe(MC, ALL, 'capital',
    'Registered capital of the business.',
    'رأس المال المسجل.',
    'SAR (assumed; the file does not say)', 'Placeholder -1 from 2025 Q2; extreme values up to 1,000,000,000,000; 0.2–8.6% missing up to 2025 Q1. Capital can change after registration.',
    'feature', 'Own column of the business', FROM_START + ' A capital increase after the start would leak success.')
describe(MC, ALL, 'city',
    'City of the registration.',
    'مدينة السجل.',
    'Arabic text', 'About 236 cities up to 2025 Q1, then 2,783–3,533 from 2025 Q2 (a much more detailed list), so city needs a cleaned common list before use. Old style contains names joined with "–".',
    'feature', 'Join key for Sama city data (after English↔Arabic mapping) and for competitor counts', FROM_START)
describe(MC, ALL, 'region',
    'Administrative region (13 regions).',
    'المنطقة الإدارية (13 منطقة).',
    'category, 13 values', '2026 Q2 shows 14 values because of the repeated header row. Monshaat writes regions with "منطقة" in front.',
    'feature', 'Join key for Monshaat (after removing "منطقة") and region-level features', FROM_START)
describe(MC, ALL, 'ecommerce',
    'Whether the business declared e-commerce activity.',
    'هل أفصح السجل عن ممارسة التجارة الإلكترونية.',
    '0 / 1', 'Three spellings of the column name. Only up to 2025 Q3; does not exist from 2025 Q4. On the website the founder must enter it.',
    'feature', 'Own column of the business', FROM_START)
describe(MC, ALL, 'start_date',
    'Registration date of the record (Gregorian).',
    'تاريخ إصدار / إنشاء السجل (ميلادي).',
    'date and time, YYYY-MM-DD HH:MM:SS', 'Placeholder 0001-01-01 from 2025 Q2; very old dates (1928). 2026 Q2 has the header text as a value.',
    'helper', 'Gives the cohort (start quarter) and the start month', 'None (known at the start)')

# ---------------------------------------------------------------- Gastat: GDP
GDP = 'gastat_gdp_2026-10-21T09_14_09.657Z.csv'
describe('Gastat', GDP, 'Quarter ID', 'Quarter as a number (year + quarter).', 'رمز الربع (السنة + رقم الربع).',
    'YYYYQ, e.g. 20101', 'None', 'helper', 'Quarter (time join)', 'Use quarters before the start quarter only')
describe('Gastat', GDP, 'Quarter', 'Quarter label, 2010 Q1 to 2026 Q2 (66 quarters).', 'الربع، من الربع الأول 2010 إلى الربع الثاني 2026.',
    'YYYY-Qn', 'None', 'helper', 'Quarter (time join)', 'Use quarters before the start quarter only')
describe('Gastat', GDP, 'Economic Activity Section ID', 'Gastat\'s own number for the economic activity (1–12).', 'رقم النشاط الاقتصادي حسب الهيئة (1–12).',
    'number 1–12', 'Not the SSIC letters: 12 groups instead of 21 sections, some merged (e.g. "Electricity, Gas and Water").', 'helper', 'Needs a mapping table to ssic_section', 'None')
describe('Gastat', GDP, 'Economic Activity Section', 'English name of the economic activity.', 'اسم النشاط الاقتصادي بالإنجليزية.',
    'English text', 'English only.', 'helper', 'Same as Section ID', 'None')
describe('Gastat', GDP, 'GDP', 'Gross domestic product of the activity in the quarter.', 'الناتج المحلي الإجمالي للنشاط في الربع.',
    'Not stated; the scale matches million SAR. Current or constant prices not stated (to confirm on the Gastat portal).', 'Unit and price basis unknown. Recent quarters may be revised later.',
    'feature', 'ssic_section (via mapping) + the quarter before the start', 'High for quarters at or after the start; use only earlier quarters (e.g. growth over the 4 quarters before the start)')

# ---------------------------------------------------------------- Gastat: SME expenses and revenues
SME = 'gastat_sme_operating_expenses_revenues_2026-10-21T09_39_24.273Z.csv'
describe('Gastat', SME, 'Economic Sectors ID', 'Economic sector as an SSIC section letter (17 sectors, B onwards).', 'القطاع الاقتصادي برمز باب التصنيف (17 قطاعاً).',
    'letter', 'Agriculture (A) and some sections are not included.', 'helper', 'Joins directly to ssic_section', 'None')
describe('Gastat', SME, 'Economic Sectors', 'English name of the sector.', 'اسم القطاع بالإنجليزية.', 'English text', 'None', 'helper', 'Same as Sectors ID', 'None')
describe('Gastat', SME, 'Establishment Size ID', 'Size class number (1 micro, 2 small, 3 medium).', 'رقم فئة الحجم (1 متناهية الصغر، 2 صغيرة، 3 متوسطة).',
    'number 1–3', 'MC has no size column; most new businesses are micro.', 'helper', 'Size class (assume micro for new businesses)', 'None')
describe('Gastat', SME, 'Establishment Size', 'Size class name.', 'اسم فئة الحجم.', 'English text', 'None', 'helper', 'Same as Size ID', 'None')
describe('Gastat', SME, 'Flow ID', 'Number of the flow (1 expenses, 2 revenues).', 'رقم نوع التدفق (1 مصروفات، 2 إيرادات).', 'number 1–2', 'None', 'helper', 'Chooses expenses or revenues', 'None')
describe('Gastat', SME, 'Flow', 'Expenses or Revenues.', 'مصروفات أو إيرادات.', 'English text', 'None', 'helper', 'Same as Flow ID', 'None')
describe('Gastat', SME, 'Year', 'Year of the figures (2019–2024).', 'سنة البيانات (2019–2024).', 'YYYY', 'Yearly only.', 'helper', 'Year (time join)', 'Use the year before the start year')
describe('Gastat', SME, 'Value', 'Total operating expenses or revenues of SMEs in the sector, size class and year.', 'إجمالي المصروفات أو الإيرادات التشغيلية للمنشآت في القطاع والحجم والسنة.',
    'Not stated (likely thousand SAR; to confirm)', 'Unit unknown. 2023 values jump for some sectors (check).',
    'feature', 'ssic_section + size + the year before the start (e.g. revenue / expense ratio)', 'High for the start year or later; use the year before')

# ---------------------------------------------------------------- Gastat: SSIC
SSIC = 'National Classification for the Economic Activities(SSIC)_15 Sep_2026_AR.xlsx'
for column, en, ar, unit in [
    ('رمز الباب', 'Section code: an Arabic letter, 21 sections mapped to A–U in their order (أ=A, ب=B … ش=T).', 'رمز الباب (حرف عربي).', '1 Arabic letter'),
    ('اسم الباب', 'Section name.', 'اسم الباب.', 'Arabic text'),
    ('رمز القسم', 'Division code.', 'رمز القسم.', '2 digits'),
    ('اسم القسم', 'Division name.', 'اسم القسم.', 'Arabic text'),
    ('رمز المجموعة', 'Group code.', 'رمز المجموعة.', '3 digits'),
    ('اسم المجموعة', 'Group name.', 'اسم المجموعة.', 'Arabic text'),
    ('رمز الفئة', 'Class code.', 'رمز الفئة.', '4 digits'),
    ('اسم الفئة', 'Class name.', 'اسم الفئة.', 'Arabic text'),
    ('رمز النشاط', 'Activity code (most detailed level).', 'رمز النشاط (أدق مستوى).', '6 digits'),
    ('مسمى النشاط', 'Activity name.', 'مسمى النشاط.', 'Arabic text'),
    ('شرح النشاط', 'Description of what the activity includes and excludes.', 'شرح ما يشمله النشاط وما لا يشمله.', 'Arabic text'),
]:
    describe('Gastat', SSIC, column, en, ar, unit,
             'Codes are stored as numbers in Excel, so leading zeros are lost unless read as text (01_preprocessing reads text).' if 'رمز' in column and column != 'رمز الباب' else 'None',
             'reference', 'ssic_section / ssic_division of each business', 'None (classification, not data about businesses)')
describe('Gastat', SSIC, 'الجهة المشرفة', 'Authority that supervises the activity.', 'الجهة المشرفة على النشاط.', 'Arabic text', 'None',
    'reference', 'Via ssic_division (possible feature: regulated activity)', 'None')
describe('Gastat', SSIC, 'الجهة المرخصة', 'Authority that licenses the activity.', 'الجهة المرخصة للنشاط.', 'Arabic text', 'None',
    'reference', 'Via ssic_division (possible feature: needs a license outside MC)', 'None')

# ---------------------------------------------------------------- Monshaat (same columns in all 4 files)
MON = 'Monshaat'
MON_LEAK = 'High: 2026 data is after every labelled cohort and includes the cohorts\' own surviving businesses. Not for the model; usable on the website.'
describe(MON, ALL, 'السنة', 'Year of the count.', 'سنة البيانات.', 'YYYY (2026 only)', 'None', 'not for model', 'Year', MON_LEAK)
describe(MON, ALL, 'الربع', 'Quarter of the count.', 'الربع.', '1 or 2', 'Only 2026 Q1 and Q2.', 'not for model', 'Quarter', MON_LEAK)
describe(MON, ALL, 'المنطقة', 'Region.', 'المنطقة.', 'Arabic text, 13 values', 'Written with "منطقة" in front (MC writes "الرياض").', 'not for model', 'region (after removing "منطقة")', MON_LEAK)
describe(MON, ALL, 'المدينة', 'City.', 'المدينة.', 'Arabic text, about 100–120 values', '12–18% of rows are "غير معرف" (unknown city).', 'not for model', 'city (after cleaning)', MON_LEAK)
describe(MON, ALL, 'النشاط الاقتصادي - المستوى الثاني', 'Economic activity at SSIC division level (name only).', 'النشاط الاقتصادي على مستوى القسم (بالاسم فقط).',
    'Arabic text, 82–88 values', 'No division code: join by name to the SSIC division name.', 'not for model', 'ssic_division (by name)', MON_LEAK)
for column, en, ar in [('عدد المنشآت متناهية الصغر', 'Number of micro enterprises.', 'عدد المنشآت متناهية الصغر.'),
                       ('عدد المنشآت الصغيرة', 'Number of small enterprises.', 'عدد المنشآت الصغيرة.'),
                       ('عدد المنشآت المتوسطة', 'Number of medium enterprises.', 'عدد المنشآت المتوسطة.'),
                       ('عدد المنشآت الكبيرة', 'Number of large enterprises.', 'عدد المنشآت الكبيرة.'),
                       ('عدد المنشآت', 'Total number of enterprises (sum of the four sizes).', 'إجمالي عدد المنشآت.')]:
    describe(MON, ALL, column, en + ' In the "New Enterprises" files: only those new in the quarter.', ar, 'count',
             'None', 'not for model', 'region + city + division', MON_LEAK)
describe(MON, ALL, '(no name, column 12)', 'Monshaat\'s disclaimer text, written in one cell next to the table.', 'نص إخلاء المسؤولية من منشآت.',
    'text', 'Not data.', 'note', 'None', 'None')

# ---------------------------------------------------------------- Sama: transactions by sector and city
SAMA_CITY = 'point-of-sale-transactions-by-sector-and-city.xlsx'
describe('Sama', SAMA_CITY, 'Starting date', 'First day of the week the figures cover.', 'تاريخ بداية الأسبوع.',
    'date, weekly, 2020-05-10 to 2025-07-06', 'Ends in July 2025.', 'helper', 'Date (time join)', 'Use weeks before the start quarter only')
describe('Sama', SAMA_CITY, 'Indicator', 'What the value measures: number of transactions (thousands), value of transactions (thousand SAR), and their % change.', 'المؤشر: عدد العمليات (بالآلاف)، قيمة العمليات (بآلاف الريالات)، ونسبة التغير.',
    'category, 4 values', 'Levels and % changes are mixed in one value column (the source of the negative values).', 'helper', 'Choose one indicator before use', 'None')
describe('Sama', SAMA_CITY, 'Sectors', 'Spending sector (e.g. clothing, hotels, food and beverage), 17 sectors + Total.', 'قطاع الإنفاق، 17 قطاعاً + الإجمالي.',
    'English text', 'Not SSIC: needs a Sama-sector → SSIC mapping. "Total" rows must not be added to the sectors.', 'helper', 'ssic_section (via mapping)', 'None')
describe('Sama', SAMA_CITY, 'City', 'City, in English capitals; 11 cities + Total.', 'المدينة بالإنجليزية؛ 11 مدينة + الإجمالي.',
    'English text', 'Only 11 cities. "Total" rows must not be added to the cities.', 'helper', 'city (English → Arabic mapping)', 'None')
describe('Sama', SAMA_CITY, 'value (Multiple units)', 'The value of the indicator for the week, sector and city.', 'قيمة المؤشر للأسبوع والقطاع والمدينة.',
    'depends on Indicator', 'Unit depends on the Indicator column.', 'feature', 'city + ssic_section + the weeks before the start', 'High for weeks at or after the start; use only earlier weeks')

# ---------------------------------------------------------------- Sama: transactions and sales by sector
SAMA_SECTOR = 'points-of-sale-transactions-and-sales-by-sector.xlsx'
describe('Sama', SAMA_SECTOR, 'Periodicity', 'Monthly, Quarterly or Annually.', 'دورية البيانات: شهري، ربعي، سنوي.',
    'category, 3 values', 'The three periodicities are mixed in one table: choose one, or the same spending is counted three times.', 'helper', 'Filter to one periodicity', 'None')
describe('Sama', SAMA_SECTOR, 'Year', 'Year (2016–2023).', 'السنة (2016–2023).', 'YYYY', 'Ends in 2023.', 'helper', 'Year (time join)', 'Low: all years are before the cohorts')
describe('Sama', SAMA_SECTOR, 'Month', 'Month for monthly rows.', 'الشهر للبيانات الشهرية.', 'JAN … DEC', 'Empty for quarterly and annual rows (29% missing).', 'helper', 'Month (time join)', 'Low')
describe('Sama', SAMA_SECTOR, 'Quarter', 'Quarter for quarterly rows.', 'الربع للبيانات الربعية.', 'Q1 … Q4', 'Empty for monthly and annual rows (77% missing).', 'helper', 'Quarter (time join)', 'Low')
describe('Sama', SAMA_SECTOR, 'Transactions / Sales', 'Actually the SECTOR (e.g. Hotels): the two column titles are swapped in the source file.', 'في الحقيقة هو القطاع: عنوانا العمودين مقلوبان في الملف.',
    'English text, 17 values', 'Header swapped with "Sector". Not SSIC: needs a mapping.', 'helper', 'ssic_section (via mapping)', 'None')
describe('Sama', SAMA_SECTOR, 'Sector', 'Actually the INDICATOR: Number of Transactions or Sales (titles swapped in the source file).', 'في الحقيقة هو المؤشر: عدد العمليات أو المبيعات.',
    'category, 2 values', 'Header swapped with "Transactions / Sales".', 'helper', 'Choose one indicator', 'None')
describe('Sama', SAMA_SECTOR, 'Value in different units', 'Value of the indicator.', 'قيمة المؤشر.',
    'Not stated (likely thousands of transactions / thousand SAR, as in the city file)', 'Unit depends on the indicator.', 'feature', 'ssic_section + year before the start (e.g. sales growth)', 'Low: ends in 2023, before every cohort (but old for 2025 cohorts)')
describe('Sama', SAMA_SECTOR, 'Date_Object', 'First day of the period as a date.', 'أول يوم في الفترة كتاريخ.', 'date', 'None', 'helper', 'Date (time join)', 'Low')
describe('Sama', SAMA_SECTOR, 'Date', 'Period label (e.g. "Q2 2023", "MAR 2023", "2023").', 'وصف الفترة.', 'text', 'Duplicates Year/Month/Quarter.', 'helper', 'None (use Date_Object)', 'None')
describe('Sama', SAMA_SECTOR, 'Name', 'Sector and indicator joined ("Hotels : Sales").', 'القطاع والمؤشر معاً.', 'text', 'Duplicates two other columns.', 'helper', 'None', 'None')

descriptions = pd.DataFrame(DESCRIPTIONS)
print(f'{len(descriptions)} descriptions written')

# Check: every profiled column has a description
mc_missing = sorted(set(profile.loc[profile['folder'] == 'MC', 'common name']) - set(descriptions.loc[descriptions['folder'] == 'MC', 'column']))
other = profile[profile['folder'] != 'MC'][['folder', 'file', 'column']].drop_duplicates()
exact = set(zip(descriptions['folder'], descriptions['file'], descriptions['column']))
any_file = set(zip(descriptions['folder'], descriptions['column'])) & {(f, c) for f, fl, c in exact if fl == ALL}
other_missing = [tuple(r) for r in other.itertuples(index=False)
                 if (r.folder, r.file, r.column) not in exact and (r.folder, r.column) not in any_file]
print('MC common names without a description:', mc_missing or 'none')
print('Other columns without a description:', other_missing or 'none')

### Cell 9 — Columns built in `01_preprocessing`
These do not exist in the raw files. They are created by Sara's preprocessing notebook; the cell numbers are
the ones in that notebook.

In [ ]:
B = 'labels (one row per business)'
M = 'mc (one row per record and quarter)'
BUILT = pd.DataFrame([
    # table, column, cell, how it is made, values, role, leakage
    (M, 'quarter', 'Cell 8', 'Quarter of the file the row comes from.', '2024Q3 … 2026Q3', 'helper', 'None'),
    (M, 'source_file', 'Cell 8', 'Name of the file the row comes from.', 'file name', 'helper', 'None'),
    (M, 'flag_no_id', 'Cell 8', 'True when the row has no ID (2025 Q3 PART1).', 'True / False', 'helper', 'None'),
    (M, 'entity_id', 'Cell 13', 'One ID per business across all quarters: the same ID = the same business; an old ID that vanished is linked to a new ID that appeared only with the same core name + city, a later start date and exactly one candidate both ways.', 'the original ID', 'ID', 'None (not a feature)'),
    (M, 'link_flag', 'Cell 13, 13b', 'How the row was linked to its business.', 'same_id, linked_new_id, ambiguous_not_linked, duplicate_across_parts, no_id, recovered_id', 'filter',
        'High if used as a feature: "re-registered under a new ID" happens after the start'),
    (M, 'id_source', 'Cell 13b', 'Where the ID of a 2025 Q3 PART1 row comes from.', 'original, recovered_from_2025Q2, not_recovered_many_matches, not_recovered_not_found, not_recovered_conflict', 'helper', 'None'),
    (M, 'recovered_id', 'Cell 13b', 'ID recovered for a 2025 Q3 PART1 row from 2025 Q2 (same name + start date + city, exactly one match).', '10 digits or empty', 'helper', 'None'),
    (M, 'record_type_source', 'Cell 13c', 'Where the record type comes from.', 'original, copied_from_2025Q2, missing', 'helper', 'None'),
    (B, 'start_day', 'Cell 14', 'Earliest start date of the business\'s IDs (the original business\'s date if it re-registered).', 'YYYY-MM-DD', 'helper', 'None'),
    (B, 'ambiguous_link', 'Cell 14', 'True when the business has an unsure old/new ID link.', 'True / False', 'helper', 'None'),
    (B, 'cohort', 'Cell 14', 'Quarter of start_day.', 'YYYYQn', 'helper (split by cohort)', 'None'),
    (B, 'one_year_quarter', 'Cell 14', 'cohort + 4 quarters: where the one-year check happens.', 'YYYYQn', 'helper', 'None'),
    (B, 'cohort_quality', 'Cell 14', 'How much the cohort can be trusted.', 'clean, start_incomplete, start_missing, start_and_check_incomplete, cannot_judge', 'filter', 'None'),
    (B, 'survived_1y', 'Cell 14', '1 = the business is still active one year after its start (under any of its IDs), 0 = it disappeared, empty = undecided.', '1 / 0 / empty', 'label', 'This IS the target'),
    (B, 'label_reason', 'Cell 14', 'Why the business got its label.', 'present_after_1_year, came_back_with_new_id, disappeared, missing_at_1_year_back_later, seen_after_1_year, disappeared_before_1_year, gone_around_1_year, cannot_judge', 'helper',
        'High: never a feature (it describes the outcome)'),
    (B, 'ssic_section', 'Cell 19', 'SSIC section from the business name (rule-based classifier, isic_classifier.py).', 'A … U or empty', 'feature', FROM_START),
    (B, 'ssic_division', 'Cell 19', 'SSIC division from the business name.', '2 digits or empty', 'feature', FROM_START),
    (B, 'ssic_label', 'Cell 19', 'Which rule matched.', 'text', 'helper', 'None'),
    (B, 'ssic_confidence', 'Cell 19', 'How sure the classifier is.', 'high, medium, low', 'helper', 'None'),
    (B, 'ssic_flag', 'Cell 19', 'Quality of the SSIC code.', 'ok, division_unclear, low_confidence, ambiguous, unclassified, names_disagree', 'filter', 'None'),
    (B, 'ssic_candidates', 'Cell 19', 'The competing sections when the name is ambiguous.', 'e.g. F41|H49', 'helper', 'None'),
    (B, 'ssic_section_name', 'Cell 19', 'Arabic name of the section (from the SSIC file).', 'Arabic text', 'helper', 'None'),
    (B, 'ssic_division_name', 'Cell 19', 'Arabic name of the division (from the SSIC file).', 'Arabic text', 'helper', 'None'),
], columns=['table', 'column', 'made in (01_preprocessing)', 'how it is made', 'values', 'role', 'leakage risk'])

with pd.option_context('display.width', 250, 'display.max_colwidth', 80):
    print(BUILT[['table', 'column', 'made in (01_preprocessing)', 'role']].to_string(index=False))

### Cell 10 — Planned features (proposal for the team)
Features the model could use, built from the columns above. **Status = proposed**: the team decides which to
build. Every one follows the leakage rule (only information from before or at the start quarter).
Two things are listed on purpose as **not to use**, with the reason.

In [ ]:
PLANNED = pd.DataFrame([
    # feature, how it is made, sources, join key, leakage note, status
    ('entity_type', 'Establishment or company, from the business\'s first quarter.', 'MC entity_type', 'own', 'Safe: first quarter', 'proposed'),
    ('legal_form_group', 'Legal form mapped to one common list (fixes the 2025 Q2 spelling change and the 2025 Q4 column swap).', 'MC legal_form / entity_type', 'own', 'Safe: first quarter', 'proposed'),
    ('log_capital, capital_missing', 'log of capital, -1 and missing set to empty + a flag, extremes capped.', 'MC capital', 'own', 'Safe: first quarter (later values may include capital increases)', 'proposed'),
    ('region', 'Region of the business.', 'MC region', 'own', 'Safe', 'proposed'),
    ('city_clean', 'City mapped to one common list (old 236 vs new 3,000+ names), or grouped by size.', 'MC city', 'own', 'Safe: first quarter', 'proposed'),
    ('ecommerce', 'Declared e-commerce (0/1).', 'MC ecommerce', 'own', 'Safe: first quarter (exists up to 2025 Q3 only)', 'proposed'),
    ('start_month', 'Month of the start date (seasonality).', 'MC start_date', 'own', 'Safe', 'proposed'),
    ('ssic_section, ssic_division', 'Sector from the name; flagged rows handled as the team decides.', 'ssic_* (Cell 19)', 'own', 'Safe', 'proposed'),
    ('competitors_city_section', 'Number of active main records with the same city and SSIC section in the start quarter. Replaces Monshaat (leakage).', 'MC + ssic_section', 'city + ssic_section + start quarter', 'Safe: counted in the start quarter', 'proposed'),
    ('new_entrants_city_section', 'Number of businesses that started in the same quarter, city and section.', 'MC + ssic_section', 'city + ssic_section + start quarter', 'Safe', 'proposed'),
    ('sector_gdp_growth', 'Growth of the sector\'s GDP over the 4 quarters before the start quarter.', 'Gastat GDP + mapping table', 'ssic_section + quarter', 'Safe only with quarters before the start', 'proposed'),
    ('sme_revenue_expense_ratio', 'Revenues / expenses of micro SMEs in the sector, the year before the start.', 'Gastat SME', 'ssic_section + year', 'Safe only with the year before the start', 'proposed'),
    ('pos_spending_city_sector', 'Value of point-of-sale transactions in the city and sector over the 13 weeks before the start, and its growth.', 'Sama city file + mappings', 'city + ssic_section + weeks', 'Safe only with weeks before the start', 'proposed'),
    ('pos_sales_sector_trend', 'Yearly growth of point-of-sale sales in the sector (last year available: 2023).', 'Sama sector file + mapping', 'ssic_section + year', 'Safe (old data)', 'proposed'),
    ('regulated_activity', 'Whether the division\'s licensing authority is outside the Ministry of Commerce.', 'SSIC file', 'ssic_division', 'Safe', 'proposed'),
    ('NOT: number of branches', 'Branches are opened after the start; counting them later uses the future.', 'MC main_id', '—', 'Leakage', 'do not use'),
    ('NOT: Monshaat counts', '2026 counts include the cohorts\' own survivors.', 'Monshaat', '—', 'Leakage (website only)', 'do not use'),
], columns=['feature', 'how it is made', 'sources', 'join key', 'leakage', 'status'])

with pd.option_context('display.width', 250, 'display.max_colwidth', 70):
    print(PLANNED[['feature', 'sources', 'join key', 'status']].to_string(index=False))

### Cell 11 — The full data dictionary
Joins the descriptions (Cell 8) with the measured facts (Cell 4). **One row per description:** MC columns are
summed up over their 18 files (original names, quarters present, range of missing share and unique values), and
the 4 Monshaat files, which have identical columns, are summed up the same way. Files whose name has no period
get it from the Phase 1 results (`FILE_PERIODS`).

Saves `data_dictionary.xlsx` in the data folder (never in Git) with the sheets `dictionary`, `built_columns`,
`planned_features`, `inventory`, `profile` and `mc_by_quarter`. Cell 12 downloads it.

In [ ]:
# Periods of the files whose name does not say it (from Phase 1, Cell 5)
FILE_PERIODS = {
    GDP: '2010Q1–2026Q2 (quarterly)',
    SME: '2019–2024 (yearly)',
    SSIC: 'version of 15 Sep 2026',
    SAMA_CITY: '2020-05-10 – 2025-07-06 (weekly)',
    SAMA_SECTOR: '2016–2023 (monthly, quarterly, yearly)',
}

def quarter_ranges(periods):
    """['2024Q3','2024Q4','2025Q2'] -> '2024Q3–2024Q4, 2025Q2'"""
    present = [q for q in QUARTERS if q in set(periods)]
    groups, run = [], []
    for q in present:
        if run and QUARTERS.index(q) != QUARTERS.index(run[-1]) + 1:
            groups.append(run)
            run = []
        run.append(q)
    if run:
        groups.append(run)
    return ', '.join(g[0] if len(g) == 1 else f'{g[0]}–{g[-1]}' for g in groups)

def span(values):
    values = values.dropna()
    return '' if values.empty else (f'{values.min():g}' if values.min() == values.max() else f'{values.min():g}–{values.max():g}')

def joined(values):
    return ' / '.join(dict.fromkeys(str(v) for v in values.dropna()))

def first_valid(values):
    values = values.dropna()
    return values.iloc[0] if len(values) else ''

def lowest(values):
    values = values.dropna()
    return values.min() if len(values) else ''

def highest(values):
    values = values.dropna()
    return values.max() if len(values) else ''

FACTS = {'sheet': ('sheet', joined),
         'kind': ('kind', lambda s: s.mode().iloc[0]),
         'missing %': ('missing %', span),
         'unique': ('unique', span),
         'min': ('min', lowest),
         'max': ('max', highest),
         'top values': ('top values', first_valid),
         'patterns': ('patterns', first_valid)}

# MC: one row per common name
mc_facts = (profile[profile['folder'] == 'MC'].groupby('common name')
            .agg(**{'original names': ('column', joined), 'periods': ('period', quarter_ranges), **FACTS})
            .reset_index().rename(columns={'common name': 'column'}).assign(folder='MC'))
mc_dict = (descriptions[descriptions['folder'] == 'MC'].merge(mc_facts, on=['folder', 'column'], how='left')
           .assign(file=f'all {(files["folder"] == "MC").sum()} MC files'))

# Other folders: one row per description (a '*' description covers every file of its folder)
others = profile[profile['folder'] != 'MC'].copy()
others['period'] = others['period'].where(others['period'] != '', others['file'].map(FILE_PERIODS))
own_description = set(zip(descriptions['folder'], descriptions['file'], descriptions['column']))
others['described as'] = [file if (folder, file, column) in own_description else ALL
                          for folder, file, column in zip(others['folder'], others['file'], others['column'])]
other_facts = (others.groupby(['folder', 'described as', 'column'], sort=False)
               .agg(**{'files': ('file', joined), 'periods': ('period', lambda s: ', '.join(dict.fromkeys(s.dropna()))),
                       **FACTS})
               .reset_index().rename(columns={'described as': 'file'}))
other_facts['original names'] = other_facts['column']
other_dict = (other_facts.merge(descriptions[descriptions['folder'] != 'MC'], on=['folder', 'file', 'column'], how='inner')
              .assign(file=lambda t: t['files']).drop(columns='files'))

DICT_ORDER = ['folder', 'file', 'sheet', 'column', 'original names', 'periods',
              'meaning (English)', 'المعنى (عربي)', 'unit / format', 'role', 'join key', 'leakage risk', 'quality issues',
              'kind', 'missing %', 'unique', 'min', 'max', 'top values', 'patterns']
dictionary = (pd.concat([mc_dict, other_dict], ignore_index=True)
              .reindex(columns=DICT_ORDER)
              .sort_values(['folder', 'file'], kind='stable')
              .reset_index(drop=True))

print(f'Data dictionary: {len(dictionary)} rows (descriptions: {len(descriptions)}) | by folder:',
      dictionary['folder'].value_counts().to_dict())
print()
print('Roles:')
print(dictionary['role'].value_counts().to_string())
print()
with pd.option_context('display.width', 250, 'display.max_colwidth', 45):
    print(dictionary[['folder', 'column', 'periods', 'role', 'join key']].to_string(index=False))

OUT = DATA_DIR / 'data_dictionary.xlsx'
with pd.ExcelWriter(OUT) as writer:
    dictionary.to_excel(writer, sheet_name='dictionary', index=False)
    BUILT.to_excel(writer, sheet_name='built_columns', index=False)
    PLANNED.to_excel(writer, sheet_name='planned_features', index=False)
    inventory.to_excel(writer, sheet_name='inventory', index=False)
    profile.to_excel(writer, sheet_name='profile', index=False)
    mc_by_quarter.to_excel(writer, sheet_name='mc_by_quarter')
print()
print('Saved', OUT)

### Cell 12 — Download the Excel files
The files are on the Colab server, not on your laptop, and are deleted when the session ends.
This cell shows a **download link** for each file (works inside VS Code). On colab.research.google.com it also
starts the download directly.

In [ ]:
import base64
from IPython.display import HTML, display

XLSX = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
for path in [DATA_DIR / 'data_dictionary.xlsx', DATA_DIR / 'data_dictionary_profile.xlsx']:
    if not path.exists():
        print('Not created yet:', path.name)
        continue
    data = base64.b64encode(path.read_bytes()).decode()
    display(HTML(f'<a download="{path.name}" href="data:{XLSX};base64,{data}">⬇ Download {path.name}</a> '
                 f'({path.stat().st_size / 1e3:.0f} KB)'))

if IN_COLAB:
    try:
        from google.colab import files as colab_files
        colab_files.download(str(DATA_DIR / 'data_dictionary.xlsx'))
    except Exception as error:
        print('Direct download is not available here; use the link above.', type(error).__name__)

### Notes — open questions for the team
- **Units** not stated in the files: Gastat GDP (likely million SAR; current or constant prices?), Gastat SME value,
  Sama values. To confirm on each source's website.
- **Mapping tables** still to make: Gastat GDP sections (1–12) → SSIC sections; Sama sectors → SSIC sections;
  Sama cities (English) → MC cities; Monshaat regions ("منطقة …") → MC regions; MC old city names → new city names.
- **For Sara (01_preprocessing):** 2026 Q2 PART1 has the header row repeated as a data row (found in Cell 13c, not
  removed); in 2025 Q4 "الكيان القانوني" holds the detailed legal form, but is mapped to `entity_type`.
- **Monshaat:** documented, not used in the model (leakage); can be used on the website.